# Human Serum aggregation

Read the source protocol CSVs for one condition and write its compact, deduplicated union.

In [ ]:
CONDITION="human_serum"
WRITE_EXPORTS=False
NOTEBOOK_ID="aggregation_base"
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML
here = Path.cwd().resolve()
ROOT = next(p for p in [here, *here.parents] if (p / "src/pep_compass").is_dir())
sys.path.insert(0, str(ROOT / "assets/tools/proteolysis_data_manager/src"))
from proteolysis_data_manager.protocols.half_life.filtering import load_half_life_data, prepare_protocol, export_protocol
from proteolysis_data_manager.integration.half_life.schema import CLEAN_COLUMNS, source_mask, row_key, typed
from proteolysis_data_manager.analysis.half_life.visualization import (formatted_table, duration_summary, plot_duration_distributions,
    plot_acceptance_conditions, plot_before_after, readable_time_axis, time_bins, DURATION_COLORS, DURATION_LABELS)
DATA_ROOT = ROOT / "data/half_time_datasets"
SERIES = ROOT / "assets/experiments/analysis/09_26/20_serum_proteolysis_calibration_max/notebooks/0_data_analysis_half_life"
RESULTS = SERIES.parents[1] / "results/0_data_analysis_half_life" / CONDITION / NOTEBOOK_ID
if WRITE_EXPORTS: RESULTS.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi":110, "axes.spines.top":False, "axes.spines.right":False, "axes.grid":True, "grid.alpha":.12})
def table(frame):
    if len(frame)>30: display(HTML(f"<p>Preview: 30 of {len(frame)} rows.</p>"))
    display(HTML(formatted_table(frame, limit=30)))
def figure(fig, name):
    if WRITE_EXPORTS: fig.savefig(RESULTS / name, bbox_inches="tight")
    plt.show(); plt.close(fig)
def counts(frame):
    point=frame.half_life_seconds.notna()
    return {"observations":len(frame), "sequences":frame.sequence.nunique(), "numeric":int(point.sum()),
            "class_only":int((~point & frame.half_life_class.notna()).sum()),
            "unresolved":int((~point & frame.half_life_class.isna()).sum())}

from proteolysis_data_manager.integration.half_life.aggregation import aggregate_environment, source_accounting
SOURCES=["peplife2","plifepred","dramp","pepmsnd","cavaco"]


## Inputs and verified measurement identity

Central manifests verify input CSVs and their shared normalization/protocol policy. A shared source reference plus identical scientific fields establishes a repeated normalized measurement. Equal sequences or equal half-lives alone do not. Missing fields were already enriched during normalization, and the complete source/ID lists survive final aggregation.

No wide provenance columns or report files are added. Conflicting measurements remain separate; their explanations are in `normalized/remarks_<source>.md`.

In [ ]:
if not WRITE_EXPORTS:
    raise RuntimeError("Template export disabled. Copy into a condition folder and enable WRITE_EXPORTS.")
before, after, lineage, possible, manifest = aggregate_environment(DATA_ROOT, CONDITION, SOURCES)
table(pd.DataFrame(manifest["inputs"]))
table(pd.DataFrame([{"condition":CONDITION,"source_input_rows":len(before),"final_measurements":len(after),"repeated_input_rows_merged":len(before)-len(after),**counts(after),"reported_ci95":int(after.ci_level.eq(.95).sum()),"unconfirmed_coincident_points":len(possible)}]))

## Source coverage, label availability and PEPlife2 overlap

Raw source representations and normalized measurements have different denominators: normalization merges verified repeated source rows. The next reduction selects species/matrix and the explicit assay mode, then applies pH/temperature gates. Tables account for each reduction.

Every retained row belongs to one label group: numerical, class-only, or unresolved. Missing half-life does not remove it. Shared sequences and verified shared measurements with PEPlife2 are reported separately; sequence overlap is not deduplication evidence.

In [ ]:
inventory=load_half_life_data(DATA_ROOT)
accounting=source_accounting(DATA_ROOT,inventory,before,after,manifest["configuration"],SOURCES)
for name in ["coverage","label_types","peplife2_overlap"]:
    print(name); table(accounting[name])
table(accounting["missingness"].pivot(index="parameter",columns="source",values="missing").reset_index())
labels=accounting["label_types"].set_index("source")
fig,ax=plt.subplots(figsize=(10,4)); bottom=pd.Series(0,index=labels.index)
for field,color,label in [("numeric","#4477AA","Numerical"),("class_only","#EE7733","Class only"),("unresolved","#999999","Unresolved")]:
    ax.bar(labels.index,labels[field],bottom=bottom,color=color,label=label);bottom+=labels[field]
ax.set(ylabel="Exported source records",title=CONDITION+" — all retained labels");ax.legend(fontsize=8)
figure(fig,"source_label_availability.png")
for _,r in accounting["label_types"].iterrows():
    assert r.numeric+r.class_only+r.unresolved==r.exported_records
for _,r in accounting["coverage"].iterrows():
    assert r.condition_records==r.selected_assay_mode+r.other_known_mode+r.unknown_assay_mode
    assert r.selected_assay_mode==r.outside_ph_temperature+r.exported_source_records

## Duration classes, distributions and missingness

**short** <30 min (blue); **medium** 30 min–6 h inclusive (orange); **long** >6 h (green); unresolved (gray). All numerical and class-only observations enter class counts. Numerical histograms use reported points or the explicitly documented CI-midpoint exception only.

Before/after counts here describe merging source copies of shared observations, not filtering out class-only data.

In [ ]:
table(pd.concat([duration_summary(before).assign(stage="Source CSV inputs"),duration_summary(after).assign(stage="Deduplicated union")],ignore_index=True))
figure(plot_duration_distributions(after),"final_distributions.png")
summary_before=duration_summary(before);summary_after=duration_summary(after);x=np.arange(len(summary_after))
fig,ax=plt.subplots(figsize=(10,4))
colors=[DURATION_COLORS[c] for c in summary_after.duration_class]
ax.bar(x-.18,summary_before.all_assigned_records,width=.36,color=colors,alpha=.35,label="Before merging")
ax.bar(x+.18,summary_after.all_assigned_records,width=.36,color=colors,label="After merging")
ax.set_xticks(x,[DURATION_LABELS[c] for c in summary_after.duration_class],rotation=25,ha="right");ax.set(ylabel="Observations");ax.legend()
figure(fig,"before_after_classes.png")
table(pd.DataFrame([{"field":f,"missing":int(after[f].isna().sum()),"denominator":len(after)} for f in CLEAN_COLUMNS if f not in ["source","source_record_id"]]))
table(lineage);table(possible)
assert not after.apply(row_key,axis=1).duplicated().any()

## Final common CSV

The result uses the same 23 fields as normalization and source protocols. Select `half_life_seconds.notna()` for numerical analysis; select `half_life_seconds.isna() & half_life_class.notna()` for class-only validation. Both can use the duration class. Unresolved labels remain present with null values. Missing SD and CI stay null.

`source` is a list of all contributing bases; `source_record_id` is the full list of source-qualified references. In CSV these two fields are JSON; the library reader decodes them.

TODO (later analysis): estimate validation tolerances separately without changing reported CI.

In [ ]:
table(pd.DataFrame({"column":CLEAN_COLUMNS}))
print("Final CSV:",DATA_ROOT/"processed"/CONDITION/(CONDITION+".csv"))